Configuración

In [0]:
import pandas as pd
import numpy as np
from scipy import stats
import os

In [0]:
# Cargar el conjunto de datos completo desde el directorio local
try:
    df_completo = pd.read_csv('prosperLoanData.csv', sep=';')
    print("¡Éxito! El dataset original ha sido cargado por completo.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'prosperLoanData.csv' en la ruta especificada.")
 
# Mostrar las dimensiones del dataset original (filas, columnas)
print(f"Dimensiones del dataset original: {df_completo.shape[0]} filas y {df_completo.shape[1]} columnas.\n")
 
# Mostrar las primeras 5 filas para inspeccionar visualmente los datos
df_completo.head()

## Limpieza inicial de datos

Se procede a realizar la limpieza inicial de loa datos, identificar y eliminar duplicados, eliminar registros sin datos de geolocalización.

In [0]:
# Eliminación de duplicados
num_duplicados = df_completo.duplicated().sum()
if num_duplicados > 0:
    df_completo.drop_duplicates(keep='first', inplace=True)
    print(f"Limpieza: Se eliminaron {num_duplicados} filas duplicadas.")

# Validación de filas y columnas
print(f"Dimensiones del dataset original: {df_completo.shape[0]} filas y {df_completo.shape[1]} columnas.\n")
 

La data no cuenta con ducplicados y se mantienen las filas originales, sin embargo, dadas las limitantes de procesamiento y el objetivo específico se realizó una clasificación de las variables siginificativas que aportarán al entrenamiento y desarrollo del modelo, por lo cual se decide conservar 11 de las 81 columnas iniciales

In [0]:
# Definir la lista de las 11 variables clave para el análisis financiero-climático
columnas_relevantes = [
    'BorrowerState', 'LoanStatus', 'EstimatedEffectiveYield',
    'BorrowerAPR', 'DebtToIncomeRatio', 'IncomeRange',
    'StatedMonthlyIncome', 'Term', 'LoanOriginalAmount',
    'MonthlyLoanPayment', 'Recommendations'
]
 
# Crear el nuevo DataFrame conservando solo las columnas seleccionadas y eliminando clientes sin
# geolocalización
df_ecoscore = df_completo.dropna(subset=['BorrowerState']).reset_index(drop=True)[columnas_relevantes].copy()


In [0]:
# Mostrar la nueva dimensión del dataset optimizado
print("--- Dataset Optimizado para EcoScore ---")
print(f"Dimensiones finales: {df_ecoscore.shape[0]} filas y {df_ecoscore.shape[1]} columnas.")
df_ecoscore.head()
# Guardar el DataFrame optimizado en un archivo CSV
df_ecoscore.to_csv('df_ecoscore.csv', index=False)

# Cálculo de Riesgo por Precipitación Extrema

Para saber el riesgo por precipitación extrema se creará variable `climate_risk_matrix` aplicando matriz de vulnerabilidad hídrica histórica por Estado, dado que es un enfoque académico los valores asignados al Índice de Riesgo por Precipitación Extrema en el modelo EcoScore fueron normalizados en una escala de 0 a 100, tomando como referencia los mapas de riesgo histórico de inundaciones de la Agencia Federal para la Gestión de Emergencias (FEMA) y los registros institucionales de eventos meteorológicos severos de la Administración Nacional Oceánica y Atmosférica (NOAA) de los Estados Unidos.

In [0]:
# Matriz de vulnerabilidad hídrica histórica por Estado (Escala 0 a 100)
climate_risk_matrix = {
    'FL': 94.2, 'LA': 91.0, 'TX': 83.5, 'NC': 77.1, 'GA': 70.4,
    'NY': 56.8, 'CA': 48.0, 'OH': 40.2, 'NV': 14.1, 'UT': 10.5
}
df_ecoscore['Precipitation_Risk_Index'] = df_ecoscore['BorrowerState'].map(climate_risk_matrix).fillna(50.0)

df_ecoscore.head(5)
